
#  C++ / Fortran 入門 --- 数値計算コードを読み書きする
# 1. このトピックのねらい
* これ以降のトピックで OpenMP による並列化を学ぶ前提として, C++ と Fortran で書かれた数値計算コードを **読み書きできる** ことを目標とする
* 全くの未経験者向けの丁寧すぎる説明はしない. 他の言語(Python など)の経験があることを前提に, 「同じことを C/C++ ではこう書く, Fortran ではこう書く」という対比で要点を押さえる
* 扱う題材
  * hello world (コンパイルと実行)
  * 数値型
  * 関数定義と呼び出し
  * 配列 (静的割り当て・動的割り当て)
  * ループ (for/do 文, while 文)
  * ファイル入出力の基本
  * コマンドライン引数処理の基本

# 2. C/C++ と Fortran --- おおまかな違い
* どちらもコンパイル型の言語. ソースを **コンパイル** して実行可能ファイルを作り, それを実行する
* C++ (拡張子 `.cpp`)
  * 文はセミコロン `;` で終わる. ブロックは `{{ ... }}` で囲む
  * 添字は **0 から** 始まる (`a[0]` 〜 `a[n-1]`)
  * 大文字小文字を区別する
* Fortran (拡張子 `.f90`)
  * 1文1行が基本 (継続は行末に `&`). ブロックは `end ...` で閉じる
  * 配列添字は既定で **1 から** 始まる (`a(1)` 〜 `a(n)`), 配列アクセスは `()` で書く
  * 大文字小文字を区別しない
  * 数値計算向けの機能(配列演算, 複素数など)が言語に組み込まれている

# 3. 環境設定
* コンパイラを使えるようにする (GPUは使わないが他のトピックと同じ設定でよい)

In [ ]:
import os
paths = os.environ["PATH"].split(":")
nvc_path = "/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin"
fj_path = "/opt/FJSVxtclanga/tcsds-1.2.41/bin"
for path in [nvc_path, fj_path]:
    if path not in paths:
        paths = [path] + paths
os.environ["PATH"] = ":".join(paths)

In [ ]:
%%bash
which nvc++
which nvfortran

# 4. hello world (コンパイルと実行)
* まずは画面に文字列を表示するだけのプログラム
* C++ では `main` 関数がプログラムの入り口. `#include <cstdio>` で `printf` が使えるようになる
* Fortran では `program 名前` 〜 `end program 名前` がプログラム本体

## 4-1. C++版

In [ ]:
%%writefile hello.cpp
#include <cstdio>

int main() {
  printf("hello, world\n");
  return 0;
}

* コンパイル. `-o` で出力する実行可能ファイル名を指定する. `-fast` は最適化オプション

In [ ]:
%%bash
nvc++ -fast hello.cpp -o hello_cpp.exe

In [ ]:
%%bash
./hello_cpp.exe

## 4-2. Fortran版

In [ ]:
%%writefile hello.f90
program hello
  print "(a)", "hello, world"
end program hello

In [ ]:
%%bash
nvfortran -fast hello.f90 -o hello_f.exe

In [ ]:
%%bash
./hello_f.exe

* 参考: Odyssey (富士通コンパイラ) では `FCCpx hello.cpp -o hello_cpp.exe` (C++), `frtpx hello.f90 -o hello_f.exe` (Fortran)
* `print "(a)", ...` の `"(a)"` は出力の書式(format)で, `a` は文字列を意味する. 数値の書式はこの後出てくる

# 5. 数値型
* 数値計算では, 整数と浮動小数点(小数), およびその精度(ビット数)を意識することが重要
* C++ の主な型: `int` (整数, 普通32bit), `long` (長い整数, 普通64bit), `float` (単精度32bit), `double` (倍精度64bit)
* Fortran では `integer(4)`, `integer(8)`, `real(4)`, `real(8)` のように **バイト数**で精度を指定するのが分かりやすい (`real(8)` が C の `double` に相当)
* 共通の注意点: **整数同士の割り算は小数部が切り捨てられる** (`7 / 2` は `3`). 小数で割りたければ少なくとも一方を浮動小数点にする

## 5-1. C++版

In [ ]:
%%writefile types.cpp
#include <cstdio>

int main() {
  int    i = 42;            /* 整数 (普通32 bit) */
  long   l = 10000000000L;  /* 長い整数 (普通64 bit) */
  float  f = 3.14f;         /* 単精度浮動小数点 (32 bit) */
  double d = 3.141592653589793;  /* 倍精度浮動小数点 (64 bit) */

  printf("i = %d\n", i);
  printf("l = %ld\n", l);
  printf("f = %f\n", f);
  printf("d = %.15f\n", d);

  /* それぞれが何バイトかを表示する */
  printf("sizeof(int)    = %zu bytes\n", sizeof(int));
  printf("sizeof(long)   = %zu bytes\n", sizeof(long));
  printf("sizeof(float)  = %zu bytes\n", sizeof(float));
  printf("sizeof(double) = %zu bytes\n", sizeof(double));

  /* 整数の割り算は切り捨て, 浮動小数点の割り算は普通の割り算 */
  printf("7 / 2     = %d\n", 7 / 2);
  printf("7.0 / 2.0 = %f\n", 7.0 / 2.0);
  return 0;
}

In [ ]:
%%bash
nvc++ -fast types.cpp -o types_cpp.exe && ./types_cpp.exe

* `%d` `%ld` `%f` などは `printf` の書式指定子で, それぞれ `int`, `long`, 浮動小数点に対応する
* Fortran では倍精度の定数を `3.14d0` のように `d0` を付けて書く (付けないと単精度扱いになり精度が落ちる)

## 5-2. Fortran版

In [ ]:
%%writefile types.f90
program types
  integer(4) :: i = 42                      ! 4バイト整数
  integer(8) :: l = 10000000000_8           ! 8バイト整数
  real(4)    :: f = 3.14                     ! 単精度 (32 bit)
  real(8)    :: d = 3.141592653589793d0      ! 倍精度 (64 bit)

  print "(a,i0)",     "i = ", i
  print "(a,i0)",     "l = ", l
  print "(a,f0.6)",   "f = ", f
  print "(a,f0.15)",  "d = ", d

  ! それぞれが何バイトかを表示する
  print "(a,i0,a)", "storage_size(i)/8 = ", storage_size(i) / 8, " bytes"
  print "(a,i0,a)", "storage_size(l)/8 = ", storage_size(l) / 8, " bytes"
  print "(a,i0,a)", "storage_size(f)/8 = ", storage_size(f) / 8, " bytes"
  print "(a,i0,a)", "storage_size(d)/8 = ", storage_size(d) / 8, " bytes"

  ! 整数同士の割り算は切り捨て, 実数の割り算は普通の割り算
  print "(a,i0)",   "7 / 2     = ", 7 / 2
  print "(a,f0.6)", "7.0 / 2.0 = ", 7.0d0 / 2.0d0
end program types

In [ ]:
%%bash
nvfortran -fast types.f90 -o types_f.exe && ./types_f.exe

* `print` の書式 `(i0)` は整数(幅は自動), `(f0.6)` は小数点以下6桁の浮動小数点を表す

# 6. 関数定義と呼び出し
* 処理をまとめて名前を付け, 引数を渡して呼び出すのが関数
* C++: `返り値の型 関数名(引数の型 引数名, ...) {{ ... return 値; }}`
* Fortran: `function 関数名(引数) result(返り値変数)` 〜 `end function`. 複数の関数は `module` にまとめ, 使う側で `use` するのが現代的な書き方
  * 引数には `intent(in)` (入力専用) などを付けると意図が明確になる

## 6-1. C++版

In [ ]:
%%writefile func.cpp
#include <cstdio>

/* 2数の平均を返す関数 (引数2つ, 返り値1つ) */
double average(double a, double b) {
  return (a + b) / 2.0;
}

/* n! を計算する関数 */
long factorial(int n) {
  long p = 1;
  for (int i = 2; i <= n; i++) {
    p = p * i;
  }
  return p;
}

int main() {
  printf("average(3.0, 4.0) = %f\n", average(3.0, 4.0));
  printf("factorial(5)      = %ld\n", factorial(5));
  return 0;
}

In [ ]:
%%bash
nvc++ -fast func.cpp -o func_cpp.exe && ./func_cpp.exe

## 6-2. Fortran版

In [ ]:
%%writefile func.f90
module func_mod
contains
  ! 2数の平均を返す関数 (引数2つ, 返り値1つ)
  function average(a, b) result(m)
    real(8), intent(in) :: a, b
    real(8) :: m
    m = (a + b) / 2.0d0
  end function average

  ! n! を計算する関数
  function factorial(n) result(p)
    integer, intent(in) :: n
    integer(8) :: p
    integer :: i
    p = 1
    do i = 2, n
       p = p * i
    end do
  end function factorial
end module func_mod

program func
  use func_mod
  print "(a,f0.6)", "average(3.0, 4.0) = ", average(3.0d0, 4.0d0)
  print "(a,i0)",   "factorial(5)      = ", factorial(5)
end program func

In [ ]:
%%bash
nvfortran -fast func.f90 -o func_f.exe && ./func_f.exe

# 7. 配列 (静的割り当て)
* 同じ型の値を並べて添字でアクセスするのが配列. 数値計算の主役
* 「静的割り当て」= 要素数をコンパイル時に決める配列
* C++: `double a[n];` (添字 0〜n-1). `n` はコンパイル時に決まる定数であること
* Fortran: `real(8) :: a(n)` (添字 1〜n). 要素数は `parameter` 定数で与える
* <font color="red">添字の起点が C は 0, Fortran は 1</font> という違いに常に注意

## 7-1. C++版

In [ ]:
%%writefile array_static.cpp
#include <cstdio>

int main() {
  /* 要素数を固定した(静的な)配列. 添字は 0 から n-1 */
  const int n = 5;
  double a[n];
  for (int i = 0; i < n; i++) {
    a[i] = i * i;        /* a[i] = i の二乗 */
  }
  double s = 0.0;
  for (int i = 0; i < n; i++) {
    printf("a[%d] = %f\n", i, a[i]);
    s += a[i];
  }
  printf("sum = %f\n", s);
  return 0;
}

In [ ]:
%%bash
nvc++ -fast array_static.cpp -o array_static_cpp.exe && ./array_static_cpp.exe

## 7-2. Fortran版

In [ ]:
%%writefile array_static.f90
program array_static
  ! 要素数を固定した(静的な)配列. 添字は既定で 1 から n
  integer, parameter :: n = 5
  real(8) :: a(n)
  real(8) :: s
  integer :: i
  do i = 1, n
     a(i) = (i - 1) * (i - 1)   ! C版の a[i]=i*i に合わせ 0,1,4,9,16
  end do
  s = 0.0d0
  do i = 1, n
     print "(a,i0,a,f0.6)", "a(", i, ") = ", a(i)
     s = s + a(i)
  end do
  print "(a,f0.6)", "sum = ", s
end program array_static

In [ ]:
%%bash
nvfortran -fast array_static.f90 -o array_static_f.exe && ./array_static_f.exe

# 8. 配列 (動的割り当て)
* 要素数を **実行時に** 決めたい場合 (例: コマンドライン引数で問題サイズを与える) は動的割り当てを使う
* C++: `double * a = (double *)malloc(sizeof(double) * n);` で確保し, 使い終わったら `free(a);` で解放する
  * `double *` は「`double` を指すポインタ」型. 配列の先頭アドレスを保持する. `a[i]` で要素にアクセスできる
* Fortran: `real(8), allocatable :: a(:)` と宣言し, `allocate(a(n))` で確保, `deallocate(a)` で解放する. C より簡潔で安全

## 8-1. C++版

In [ ]:
%%writefile array_dynamic.cpp
#include <cstdio>
#include <cstdlib>

int main(int argc, char ** argv) {
  /* 要素数を実行時に決める(動的な)配列 */
  long n = (argc > 1 ? atol(argv[1]) : 5);
  /* malloc で n 要素分の領域を確保する (calloc は 0 初期化付き) */
  double * a = (double *)malloc(sizeof(double) * n);
  for (long i = 0; i < n; i++) {
    a[i] = 1.0 / (i + 1);     /* 1/1, 1/2, 1/3, ... */
  }
  double s = 0.0;
  for (long i = 0; i < n; i++) {
    s += a[i];
  }
  printf("sum of 1/k (k=1..%ld) = %f\n", n, s);
  free(a);                    /* 使い終わったら解放する */
  return 0;
}

In [ ]:
%%bash
nvc++ -fast array_dynamic.cpp -o array_dynamic_cpp.exe && ./array_dynamic_cpp.exe 100

## 8-2. Fortran版

In [ ]:
%%writefile array_dynamic.f90
program array_dynamic
  ! 要素数を実行時に決める(動的な)配列
  character(len=32) :: arg
  integer(8) :: n, i
  real(8), allocatable :: a(:)
  real(8) :: s
  n = 5
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  allocate(a(n))               ! n 要素分の領域を確保する
  do i = 1, n
     a(i) = 1.0d0 / i          ! 1/1, 1/2, 1/3, ...
  end do
  s = 0.0d0
  do i = 1, n
     s = s + a(i)
  end do
  print "(a,i0,a,f0.6)", "sum of 1/k (k=1..", n, ") = ", s
  deallocate(a)                ! 使い終わったら解放する
end program array_dynamic

In [ ]:
%%bash
nvfortran -fast array_dynamic.f90 -o array_dynamic_f.exe && ./array_dynamic_f.exe 100

# 9. ループ (for / do 文, while 文)
* 繰り返しはループで書く. 数値計算では配列の各要素を順に処理するのに多用する
* C++: `for (初期化; 条件; 更新) {{ ... }}`, 条件で繰り返す `while (条件) {{ ... }}`
* Fortran: `do 変数 = 始点, 終点 [, 増分]` 〜 `end do`, 条件で繰り返す `do while (条件)` 〜 `end do`
* (これ以降のトピックで OpenMP が並列化するのは, 主にこの `for`/`do` ループである)

## 9-1. C++版

In [ ]:
%%writefile loop.cpp
#include <cstdio>

int main() {
  /* for 文: 初期化; 条件; 更新 */
  printf("for loop:\n");
  for (int i = 0; i < 5; i++) {
    printf("  i = %d\n", i);
  }

  /* while 文: 条件が成り立つ間繰り返す */
  printf("while loop:\n");
  int j = 1;
  long p = 1;
  while (p < 100) {
    p = p * 2;     /* 2 の累乗 */
    printf("  2^%d = %ld\n", j, p);
    j++;
  }
  return 0;
}

In [ ]:
%%bash
nvc++ -fast loop.cpp -o loop_cpp.exe && ./loop_cpp.exe

## 9-2. Fortran版

In [ ]:
%%writefile loop.f90
program loop
  integer :: i, j
  integer(8) :: p
  ! do 文: 始点, 終点 (, 増分)
  print "(a)", "do loop:"
  do i = 0, 4
     print "(a,i0)", "  i = ", i
  end do

  ! do while 文: 条件が成り立つ間繰り返す
  print "(a)", "do while loop:"
  j = 1
  p = 1
  do while (p < 100)
     p = p * 2          ! 2 の累乗
     print "(a,i0,a,i0)", "  2^", j, " = ", p
     j = j + 1
  end do
end program loop

In [ ]:
%%bash
nvfortran -fast loop.f90 -o loop_f.exe && ./loop_f.exe

# 10. ファイル入出力の基本
* 計算結果をファイルに保存したり, 入力データを読み込んだりする
* C++: `fopen` で開き (`"w"`=書き込み, `"r"`=読み込み), `fprintf`/`fscanf` で書き/読み, `fclose` で閉じる
* Fortran: `open(unit=番号, file=..., ...)` で開き, `write`/`read` で書き/読み, `close` で閉じる. 読み込みの終端は `iostat` で判定する

## 10-1. C++版

In [ ]:
%%writefile fileio.cpp
#include <cstdio>

int main() {
  /* 書き込み: "w" で開く */
  FILE * wp = fopen("data.txt", "w");
  if (wp == NULL) { printf("cannot open for write\n"); return 1; }
  for (int i = 0; i < 5; i++) {
    fprintf(wp, "%d %f\n", i, i * 0.5);   /* 1行に2つの数を書く */
  }
  fclose(wp);

  /* 読み込み: "r" で開く */
  FILE * rp = fopen("data.txt", "r");
  if (rp == NULL) { printf("cannot open for read\n"); return 1; }
  int i;
  double x;
  /* fscanf が 2 個読めた間繰り返す */
  while (fscanf(rp, "%d %lf", &i, &x) == 2) {
    printf("read: i = %d, x = %f\n", i, x);
  }
  fclose(rp);
  return 0;
}

In [ ]:
%%bash
nvc++ -fast fileio.cpp -o fileio_cpp.exe && ./fileio_cpp.exe && echo "--- data.txt ---" && cat data.txt

## 10-2. Fortran版

In [ ]:
%%writefile fileio.f90
program fileio
  integer :: i, ios
  real(8) :: x
  ! 書き込み: unit 番号を割り当てて開く
  open(unit=10, file="data.txt", status="replace", action="write")
  do i = 0, 4
     write(10, "(i0,1x,f0.6)") i, i * 0.5d0   ! 1行に2つの数を書く
  end do
  close(10)

  ! 読み込み
  open(unit=10, file="data.txt", status="old", action="read")
  do
     read(10, *, iostat=ios) i, x      ! iostat /= 0 で読み終わり
     if (ios /= 0) exit
     print "(a,i0,a,f0.6)", "read: i = ", i, ", x = ", x
  end do
  close(10)
end program fileio

In [ ]:
%%bash
nvfortran -fast fileio.f90 -o fileio_f.exe && ./fileio_f.exe && echo "--- data.txt ---" && cat data.txt

# 11. コマンドライン引数処理の基本
* 問題サイズやパラメータを, 実行時にコマンドラインから渡せると便利 (これまでのトピックでも `./a.exe 72 100000000` のように使ってきた)
* C++: `int main(int argc, char ** argv)` の `argc` が引数の個数(プログラム名を含む), `argv[i]` が i 番目の引数(文字列). `atoi`/`atof` で数値に変換する
* Fortran: `command_argument_count()` が引数の個数(プログラム名を含まない), `get_command_argument(i, 変数)` で i 番目を取り出す (i=0 はプログラム名). 数値変換は内部 `read` で行う

## 11-1. C++版

In [ ]:
%%writefile args.cpp
#include <cstdio>
#include <cstdlib>

int main(int argc, char ** argv) {
  /* argc は引数の個数 (プログラム名自身を含む)
     argv[0] はプログラム名, argv[1], argv[2], ... が実際の引数(文字列) */
  printf("argc = %d\n", argc);
  for (int i = 0; i < argc; i++) {
    printf("argv[%d] = %s\n", i, argv[i]);
  }
  /* 文字列を数値に変換する: atoi (整数), atof (浮動小数点) */
  int    n = (argc > 1 ? atoi(argv[1]) : 10);
  double x = (argc > 2 ? atof(argv[2]) : 1.0);
  printf("n = %d, x = %f\n", n, x);
  return 0;
}

In [ ]:
%%bash
nvc++ -fast args.cpp -o args_cpp.exe && ./args_cpp.exe 5 2.5

## 11-2. Fortran版

In [ ]:
%%writefile args.f90
program args
  character(len=256) :: arg
  integer :: argc, i, n
  real(8) :: x
  ! command_argument_count() は引数の個数 (プログラム名を含まない)
  argc = command_argument_count()
  print "(a,i0)", "argc = ", argc
  do i = 0, argc
     call get_command_argument(i, arg)   ! i=0 はプログラム名
     print "(a,i0,a,a)", "arg(", i, ") = ", trim(arg)
  end do
  ! 文字列を数値に変換する: 内部 read を使う
  n = 10
  x = 1.0d0
  if (argc >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  if (argc >= 2) then
     call get_command_argument(2, arg); read (arg, *) x
  end if
  print "(a,i0,a,f0.6)", "n = ", n, ", x = ", x
end program args

In [ ]:
%%bash
nvfortran -fast args.f90 -o args_f.exe && ./args_f.exe 5 2.5

# 12. まとめ
* これで, このあとのトピックで出てくる数値計算コードを読み書きするのに必要な C++ / Fortran の基本要素が一通り揃った
* 特に重要な対比
  * 配列添字: C は 0 起点, Fortran は 1 起点
  * 動的配列: C は `malloc`/`free` + ポインタ, Fortran は `allocatable` + `allocate`/`deallocate`
  * 並列化の主対象となるループ: C の `for`, Fortran の `do`
* 次のトピックからは, これらのループを OpenMP で並列化していく